# webGrainy tutorial videos → Google Drive

This notebook records the **webGrainy tutorial videos** (1280×720 MP4) from the app's source code and saves them to your Google Drive, together with a **timestamped transcript of every caption and title card**, ready for writing or generating a voiceover.

For each tutorial `NN-name` it saves:

| File | What it is |
| --- | --- |
| `NN-name.mp4` | the video: a webGrainy logo intro, the tutorial, and a logo outro naming the next video |
| `NN-name.srt` / `.vtt` | the on-screen text as subtitles (start → end of each caption) |
| `NN-name.transcript.md` | the same as a readable table: start, end, seconds available, word count, text |
| `NN-name.transcript.json` | the same, for scripts (e.g. text-to-speech) |
| `ALL_TRANSCRIPTS.md` | every transcript in one file, in tutorial order |

Captions stay on screen long enough to be **read aloud at about 2.5 words per second** (150 words a minute), so a voiceover of the caption text fits its slot. Change `WORDS_PER_SECOND` below to match your narrator.

**How to use:** *Runtime → Run all*. A standard CPU runtime is enough (the browser runs the model on the CPU). Recording takes about 2–4 minutes per video, so all 25 take roughly 1–1½ hours. Videos already in your Drive folder are skipped, so if the runtime disconnects, just run all again and it carries on where it stopped. Tutorial 25 opens the results in RustGIS (web.rustgis.app), so the runtime needs normal internet access (Colab has it).

## 1. Settings

In [ ]:
#@title Settings { display-mode: "form" }
REPO_URL = "https://github.com/LukeAFullard/webGrainy.git"  #@param {type:"string"}
#@markdown Branch with the tutorial scripts (use `main` once they are merged).
BRANCH = "claude/upbeat-heisenberg-50l4dm"  #@param {type:"string"}
#@markdown Folder in *My Drive* where videos and transcripts are saved.
DRIVE_FOLDER = "webGrainy tutorials"  #@param {type:"string"}
#@markdown Tutorials to record: blank for all, or numbers separated by spaces, e.g. `01 05 12`.
TUTORIALS = ""  #@param {type:"string"}
#@markdown Narration speed the captions are timed for (words per second; 2.5 ≈ 150 words a minute).
WORDS_PER_SECOND = 2.5  #@param {type:"number"}
#@markdown Re-record videos that are already in the Drive folder.
OVERWRITE = False  #@param {type:"boolean"}

## 2. Connect Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
OUT_DIR = os.path.join('/content/drive/MyDrive', DRIVE_FOLDER)
os.makedirs(OUT_DIR, exist_ok=True)
print('Saving to', OUT_DIR)

## 3. Install the tools and build the app

Installs Node.js 22, ffmpeg, the project's packages and the Chromium browser that Playwright drives, then builds the app. Takes about 3–5 minutes.

In [ ]:
import subprocess, shutil, sys

def sh(cmd, cwd=None):
    """Runs a shell command, streaming its output; stops the notebook if it fails."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'command failed ({p.returncode}): {cmd}')

# Node.js 22 (the app needs 22.13 or later).
node_ok = False
if shutil.which('node'):
    v = subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    major, minor = (int(x) for x in v.lstrip('v').split('.')[:2])
    node_ok = (major == 22 and minor >= 13) or major >= 24
if not node_ok:
    sh('curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null')
sh('node --version && npm --version')
if not shutil.which('ffmpeg') or not shutil.which('unzip'):
    sh('apt-get update -qq && apt-get install -y -qq ffmpeg unzip > /dev/null')

REPO = '/content/webGrainy'
if os.path.isdir(REPO):
    sh(f'git fetch --depth 1 origin {BRANCH} && git checkout -q FETCH_HEAD', cwd=REPO)
else:
    sh(f'git clone -q --depth 1 --branch {BRANCH} {REPO_URL} {REPO}')
sh('git log -1 --format="%h %s"', cwd=REPO)
sh('npm ci --no-audit --no-fund', cwd=REPO)
sh('npx playwright install --with-deps chromium', cwd=REPO)
sh('npm run build', cwd=REPO)

## 4. Record the videos

Each tutorial is recorded on its own, then its video and transcripts are copied to Drive straight away. The first one also downloads the U-Net model (26 MB) and the two sample photos.

In [ ]:
import glob, re, time

tut_dir = os.path.join(REPO, 'scripts', 'tutorials')
names = sorted(f[:-4] for f in os.listdir(tut_dir) if re.match(r'^\d\d-.*\.mjs$', f))
wanted = TUTORIALS.split()
if wanted:
    names = [n for n in names if any(n.startswith(w.zfill(2)) for w in wanted)]
print(len(names), 'tutorials:', ', '.join(names))

local_out = os.path.join(REPO, 'tutorials', 'out')
env = dict(os.environ, TUTORIAL_WPS=str(WORDS_PER_SECOND))
done, failed = [], []
for name in names:
    if not OVERWRITE and os.path.exists(os.path.join(OUT_DIR, name + '.mp4')):
        print(f'\n=== {name}: already in Drive, skipped')
        continue
    print(f'\n=== {name}', flush=True)
    t0 = time.time()
    p = subprocess.Popen(['node', 'scripts/record-tutorials.mjs', name[:2]], cwd=REPO, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print('  ' + line, end='', flush=True)
    ok = p.wait() == 0 and os.path.exists(os.path.join(local_out, name + '.mp4'))
    if not ok:
        failed.append(name)
        err = os.path.join(local_out, name + '.error.png')
        if os.path.exists(err):
            shutil.copy(err, OUT_DIR)
        continue
    for ext in ('.mp4', '.srt', '.vtt', '.transcript.md', '.transcript.json'):
        shutil.copy(os.path.join(local_out, name + ext), OUT_DIR)
    done.append(name)
    print(f'  saved to Drive ({time.time() - t0:.0f} s)', flush=True)

print('\nRecorded:', ', '.join(done) or 'none')
if failed:
    print('FAILED:', ', '.join(failed), '(a screenshot NN-name.error.png of the moment it failed is in the Drive folder; run this cell again to retry)')

## 5. One transcript file for all videos

In [ ]:
import json
parts = ['# webGrainy tutorial videos: on-screen text with timings', '',
         f'Captions are timed for a voiceover at about {WORDS_PER_SECOND} words a second. '
         'Each row: start and end (minutes:seconds), the time available, the word count and the text. '
         'Rows marked [title], [card], [image] or [end] are full-screen cards; the others are captions.', '']
for f in sorted(glob.glob(os.path.join(OUT_DIR, '*.transcript.md'))):
    parts.append(open(f, encoding='utf-8').read())
with open(os.path.join(OUT_DIR, 'ALL_TRANSCRIPTS.md'), 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(parts))

total = 0
for f in sorted(glob.glob(os.path.join(OUT_DIR, '*.transcript.json'))):
    t = json.load(open(f, encoding='utf-8'))
    total += t['duration']
    print(f"{t['name']:<38} {t['duration'] / 60:5.1f} min  {len(t['segments']):3d} captions")
print(f'\nTotal: {total / 60:.0f} minutes of video in {OUT_DIR}')

## 6. Preview a video

In [ ]:
#@title Preview { display-mode: "form" }
PREVIEW = "01"  #@param {type:"string"}
from IPython.display import HTML
from base64 import b64encode
match = sorted(glob.glob(os.path.join(OUT_DIR, PREVIEW.zfill(2) + '-*.mp4')))
if match:
    data = b64encode(open(match[0], 'rb').read()).decode()
    display(HTML(f'<video width=800 controls src="data:video/mp4;base64,{data}"></video>'))
else:
    print('No video', PREVIEW, 'in', OUT_DIR)